# EXECUTED OFFLINE DEMO — 31 source rows

# Tennis Match History Project
**Team:** Phouvadeth Vathana, Joseph Bronyah, Dhrumil Patel

This notebook runs the code in `tennis_project.py` and `tennis_analyses.py`.
Keep both files in the same folder as this notebook.

**Questions:** performance after ranking upsets; repeat-opponent history;
performance against left-handed opponents.

The output is observational evidence, not proof of psychological recovery or causation.
Read `README.md` for the scope, match-order approximation, and model definitions.

## 1. Install dependencies
Run `python -m pip install -r requirements.txt` in your terminal before opening the notebook.
The cell below is an optional alternative for an existing notebook environment.

In [1]:
# Uncomment if needed, then restart the kernel after installation.
# %pip install -r requirements.txt

In [2]:
from pathlib import Path
import json
import logging
import pandas as pd
from IPython.display import display, Image
from tennis_project import Config, run_project

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
assert Path("tennis_analyses.py").exists(), "Open this notebook from the extracted project folder."

## 2. Choose the full study or the offline demonstration

`RUN_FULL_STUDY = True` downloads the pinned annual ATP files. The default main
period is 2000–2024, with 1990–1999 for earlier history. Change the configuration
before running to adjust the dates or upset definition.

`False` runs **31 real source rows from Brisbane 2024**. This checks code only;
it is not enough data to answer the research questions. The models will be skipped.

Default ranking upset: a top-50 player loses to someone at least 30 ranking
places lower. This is a chosen ranking-based definition, not betting odds.

In [3]:
RUN_FULL_STUDY = False

config = Config(
    history_start=1990,
    analysis_start=2000,
    end_year=2024,
    test_start=2020,
    upset_max_rank=50,
    upset_rank_gap=30,
    min_hand_matches=10,
    bootstrap_samples=500,
    output_dir="outputs",
)
input_files = None
if not RUN_FULL_STUDY:
    config.history_start = config.analysis_start = config.end_year = 2024
    config.output_dir = "demo_outputs"
    input_files = [Path("tests/fixtures/brisbane_2024_excerpt.csv")]
    print("DEMONSTRATION ONLY: not the full research dataset.")

DEMONSTRATION ONLY: not the full research dataset.


## 3. Download, validate, build features, and analyze
The program caches original files and records hashes. Failed downloads are not
silently skipped. A rerun replaces known generated outputs in the selected
output directory; use a new directory to preserve an earlier configuration.

In [4]:
summary = run_project(config, input_files=input_files)
output = Path(config.output_dir)
print("Raw match rows:", summary["raw_match_rows"])
print("Eligible analysis matches:", summary["eligible_analysis_matches"])
for question in ["q1", "q2", "q3"]:
    print(question.upper(), summary[question]["status"])

INFO: Loaded 31 raw match rows from 1 file(s).


INFO: Saved results in /mnt/data/tennis_final_project/demo_outputs


Raw match rows: 31
Eligible analysis matches: 29
Q1 Descriptive Q1 tables saved; insufficient follow-up, players, or upset variation for the adjusted model.
Q2 Q2 history features saved; insufficient earlier/later data or outcome variation for a model comparison.
Q3 Q3 descriptive tables saved; insufficient players with both opponent hands and complete covariates for adjustment.


## 4. Check data quality before interpreting results
Counts refer to physical matches. The feature table has two player views per
match, so its row count is not a count of independent matches.

The program screens incomplete results and ambiguity, but does not independently
verify exact match dates or every tennis score.

In [5]:
display(pd.read_csv(output / "coverage_by_year.csv"))
display(pd.read_csv(output / "missing_values.csv"))
display(pd.read_csv(output / "player_row_quality.csv"))
print("Exclusion flags (overlapping):")
print(json.dumps(summary["exclusion_flag_counts_nonexclusive"], indent=2))

,season,source_matches,eligible_matches,unknown_winner_hands,unknown_loser_hands
0,2024,31,29,0,0


,field,missing_count,missing_percent
0,tourney_id,0,0.000000
1,tourney_name,0,0.000000
2,tourney_date,0,0.000000
3,tourney_level,0,0.000000
4,round,0,0.000000
5,match_num,0,0.000000
6,surface,0,0.000000
7,best_of,0,0.000000
8,score,0,0.000000
9,winner_id,0,0.000000


,flag,count
0,unknown_player_hand_rows,0
1,missing_player_rank_rows,0
2,invalid_or_missing_service_stat_rows,2


Exclusion flags (overlapping):
{
  "invalid_player_ids": 0,
  "conflicting_match_key": 0,
  "outside_individual_AMG_scope": 0,
  "unfinished_or_unrecognized_score": 2,
  "ambiguous_order": 0,
  "outside_history_window": 0,
  "missing_match_key": 0
}


In [6]:
features = pd.read_csv(output / "player_match_features.csv.gz")
columns = ["match_id", "player_name", "opponent_name", "won", "eligible",
           "player_age", "player_hand", "opponent_hand", "prior_matches",
           "recent_win_rate", "prior_h2h_meetings", "prior_h2h_wins"]
display(features[columns].head(12))

,match_id,player_name,opponent_name,won,eligible,player_age,player_hand,opponent_hand,prior_matches,recent_win_rate,prior_h2h_meetings,prior_h2h_wins
0,2024-0339:270,Andy Murray,Grigor Dimitrov,0,True,36.6,R,R,0.0,NaN,0.0,0.0
1,2024-0339:270,Grigor Dimitrov,Andy Murray,1,True,32.6,R,R,0.0,NaN,0.0,0.0
2,2024-0339:271,Li Tu,Daniel Altmaier,0,True,27.5,R,R,0.0,NaN,0.0,0.0
3,2024-0339:271,Daniel Altmaier,Li Tu,1,True,25.3,R,R,0.0,NaN,0.0,0.0
4,2024-0339:272,Thanasi Kokkinakis,Rinky Hijikata,0,True,27.7,R,R,0.0,NaN,0.0,0.0
5,2024-0339:272,Rinky Hijikata,Thanasi Kokkinakis,1,True,22.8,R,R,0.0,NaN,0.0,0.0
6,2024-0339:273,Tomas Martin Etcheverry,Tomas Machac,0,True,24.4,R,R,0.0,NaN,0.0,0.0
7,2024-0339:273,Tomas Machac,Tomas Martin Etcheverry,1,True,23.2,R,R,0.0,NaN,0.0,0.0
8,2024-0339:274,Ugo Humbert,Alex Michelsen,1,True,25.5,L,R,0.0,NaN,0.0,0.0
9,2024-0339:274,Alex Michelsen,Ugo Humbert,0,True,19.3,R,L,0.0,NaN,0.0,0.0


## 5. Question 1: performance after ranking upsets
Age and experience are separate variables. Experience counts earlier eligible
recorded matches, not the entire professional career.

A missing follow-up window is not zero wins. The next-three-match rate is
calculated only when all three next recorded matches are eligible. We do not
skip a retirement or an ambiguous event to find later favorable observations.

The following summaries are **unadjusted**. The optional adjusted rate model
uses a common rank-eligible population and accounts for observed strength,
recent form, age, experience, surfaces, format, and season. Its results are
exploratory associations, with focal-player-clustered uncertainty.

In [7]:
display(pd.read_csv(output / "q1_summary_age.csv"))
display(pd.read_csv(output / "q1_summary_experience.csv"))
display(pd.read_csv(output / "q1_threshold_sensitivity.csv"))
print(json.dumps(summary["q1"], indent=2))

,loss_type,age_group,index_losses,unique_players,usable_next_matches,next_match_win_rate,complete_three_match_windows,next_three_win_rate,mean_age,mean_prior_matches
0,other_ranked_loss,under_23,6,6,0,NaN,0,NaN,21.050000,1.333333
1,other_ranked_loss,23_to_29,14,14,0,NaN,0,NaN,26.014286,0.714286
2,other_ranked_loss,30_plus,7,7,0,NaN,0,NaN,32.971429,0.714286
3,ranking_upset,23_to_29,2,2,0,NaN,0,NaN,23.700000,0.000000


,loss_type,experience_group,index_losses,unique_players,usable_next_matches,next_match_win_rate,complete_three_match_windows,next_three_win_rate,mean_age,mean_prior_matches
0,other_ranked_loss,under_100,27,27,0,NaN,0,NaN,26.714815,0.851852
1,ranking_upset,under_100,2,2,0,NaN,0,NaN,23.700000,0.000000


,rank_gap_threshold,age_group,index_losses,complete_windows,next_three_win_rate
0,20,under_23,1,0,NaN
1,20,23_to_29,3,0,NaN
2,30,23_to_29,2,0,NaN
3,50,23_to_29,1,0,NaN


{
  "index_losses": 29,
  "ranking_upsets": 2,
  "complete_three_match_followups": 0,
  "status": "Descriptive Q1 tables saved; insufficient follow-up, players, or upset variation for the adjusted model.",
  "adjusted_model_eligible_events": 0
}


In [8]:
q1_coefficients = output / "q1_adjusted_coefficients.csv"
if q1_coefficients.exists():
    display(pd.read_csv(q1_coefficients))
else:
    print("Q1 adjustment was not fitted. See the status above; no result has been invented.")

Q1 adjustment was not fitted. See the status above; no result has been invented.


## 6. Question 2: do previous meetings improve predictions?
Compare the baseline, baseline plus meeting count, and baseline plus head-to-head
results. They use the same earlier-season training set and later-season test set.

**Lower Brier score and log loss are better.** In the paired Brier comparison,
a positive improvement means the full head-to-head model had lower error than
the baseline. An interval spanning zero does not provide a clear direction.

Earlier test outcomes can update later test histories, but the prediction
models remain frozen. No current-match statistics or future outcomes are predictors.
Opponent-pair bootstrap intervals do not capture all shared-player dependence.

In [9]:
metrics_file = output / "q2_model_metrics.csv"
if metrics_file.exists():
    metrics = pd.read_csv(metrics_file)
    display(metrics.loc[metrics["meeting_group"].eq("all")])
    display(metrics.loc[~metrics["meeting_group"].eq("all")])
else:
    print("Insufficient data for the chronological model comparison.")
print(json.dumps(summary["q2"], indent=2))

Insufficient data for the chronological model comparison.
{
  "status": "Q2 history features saved; insufficient earlier/later data or outcome variation for a model comparison.",
  "training_matches": 0,
  "test_matches": 29,
  "test_start": 2020,
  "orientation": "Player A has the smaller numeric player ID, independent of match outcome."
}


## 7. Question 3: left-handed versus right-handed opponents
The paired table contains the **same player's** L-minus-R differences. Raw
service-point rates and win-rate differences are unadjusted and include counts.
A positive raw difference means better recorded performance against left-handers.

The optional within-player model adjusts match-result comparisons. Its
`opponent_left` coefficient describes right-handed focal players; add the
interaction for left-handed focal players. A coefficient of 0.02 means about
two percentage points, not 2% relative growth. This is not a causal estimate.

In [10]:
by_hand = pd.read_csv(output / "q3_by_player_and_hand.csv")
paired = pd.read_csv(output / "q3_paired_player_comparison.csv")
display(by_hand.head(12))
display(paired.head(12))
coeff_file = output / "q3_adjusted_coefficients.csv"
if coeff_file.exists():
    coefficients = pd.read_csv(coeff_file)
    display(coefficients.loc[coefficients["term"].str.contains("opponent_left")])
else:
    print("Insufficient data for the within-player adjusted model.")
print(json.dumps(summary["q3"], indent=2))

,player_id,opponent_hand,matches,wins,win_rate,service_matches,service_points_won,service_points_observed,service_win_rate
0,104745,R,3,2,0.666667,3,145.0,200.0,0.725000
1,104918,R,1,0,0.000000,1,53.0,86.0,0.616279
2,105777,R,5,5,1.000000,5,232.0,314.0,0.738854
3,105870,R,2,1,0.500000,2,107.0,162.0,0.660494
4,105902,R,3,2,0.666667,3,154.0,220.0,0.700000
5,105916,R,1,0,0.000000,1,77.0,101.0,0.762376
6,106186,L,1,0,0.000000,1,22.0,46.0,0.478261
7,106233,L,1,0,0.000000,1,33.0,54.0,0.611111
8,106331,R,1,0,0.000000,1,82.0,127.0,0.645669
9,106423,R,1,0,0.000000,1,42.0,62.0,0.677419


,player_id,matches_L,matches_R,win_rate_L,win_rate_R,service_win_rate_L,service_win_rate_R,win_rate_difference_L_minus_R,service_rate_difference_L_minus_R,player_name


Insufficient data for the within-player adjusted model.
{
  "status": "Q3 descriptive tables saved; insufficient players with both opponent hands and complete covariates for adjustment.",
  "player_match_rows": 58,
  "unique_matches": 29,
  "players_meeting_raw_comparison_minimum": 0
}


## 8. Exploratory charts
The recovery and paired handedness charts are unadjusted descriptions. Read
the opportunity counts and adjusted outputs before interpreting them.

In [11]:
chart_files = sorted((output / "charts").glob("*.png"))
if not chart_files:
    print("No charts available for this sample.")
for path in chart_files:
    display(Image(filename=str(path)))

No charts available for this sample.


## 9. Before writing the report
Report actual source and eligible match counts, missing information, exclusions,
and the chosen upset definition. Keep age and experience separate. Describe
match order as inferred from tournament week and knockout round, not verified
match-day dates. Label early-history counts as observed history rather than
complete careers. Report follow-up censoring and uncertainty.

`SOURCES.md` and `DATA_LICENSE.txt` contain attribution. `run_summary.json`
records configuration, model status, versions, and input hashes.

The bundled real sample and synthetic test fixtures verify implementation;
they do not establish the answers to your research questions.

In [12]:
# Optional: run the automated implementation checks.
# !python -m pytest -q